# **Population: Sexes by age in Ireland**

## **Part 1: Differences between sexes by age in Ireland**


In [1]:
# Import necessary libraries
import pandas as pd # For calculation and data manipulation
import matplotlib.pyplot as plt # For data visualisation
import seaborn as sns   # For enhanced data visualisation
import numpy as np      # For data processing

# Confirmation message
print("Libraries loaded successfully!")


Libraries loaded successfully!


In [2]:
# Define file path and name we will be looking at 
FILENAME = "cso_populationbyageandsex.csv"
DATADIR = "data/"
FULLPATH = DATADIR + FILENAME

# Read the CSV file into a Data Frame
df = pd.read_csv(FULLPATH)


In [3]:
# Drop all unnecessary columns
drop_col_list = ["Statistic Label","CensusYear","UNIT"]
df.drop(columns=drop_col_list, inplace=True)

**Before we get cracking with the task, we'll clean the data, so we can perform the analysis easily.**


In [4]:
# Removing all ages
df = df[df["Single Year of Age"] != "All ages"]

# Changing the columns using a find and replace
df ["Single Year of Age"] = df ["Single Year of Age"].astype(str).str.replace("Under 1 year","0")
df ["Single Year of Age"] = df ["Single Year of Age"].astype(str).str.replace("100 years and over","100")

# Removing all non-digit characters from the selected column
df["Single Year of Age"] = df["Single Year of Age"].str.replace(r"\D", "", regex=True)

# Make sure the column is an integer for plotting later
df["Single Year of Age"] = df["Single Year of Age"].astype("int64")

# Remove Ireland as an Administrative County
df = df[df["Administrative Counties"] != "Ireland"]

# Check the general info of the dataframe 
print(df.info())

# Display the first and last 5 rows of data, to make sure the edits have worked
display(df.head())
display(df.tail())

# Source: Class notes
# Source: http://medium.com/@will4856/basic-steps-when-cleaning-a-data-set-using-pandas-3576e716173d


<class 'pandas.core.frame.DataFrame'>
Index: 9393 entries, 33 to 9791
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Sex                      9393 non-null   object
 1   Single Year of Age       9393 non-null   int64 
 2   Administrative Counties  9393 non-null   object
 3   VALUE                    9393 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 366.9+ KB
None


,Sex,Single Year of Age,Administrative Counties,VALUE
33,Both sexes,0,Carlow County Council,699
34,Both sexes,0,Dublin City Council,6213
35,Both sexes,0,Dún Laoghaire Rathdown County Council,2457
36,Both sexes,0,Fingal County Council,4009
37,Both sexes,0,South Dublin County Council,3544


,Sex,Single Year of Age,Administrative Counties,VALUE
9787,Female,100,Roscommon County Council,7
9788,Female,100,Sligo County Council,9
9789,Female,100,Cavan County Council,12
9790,Female,100,Donegal County Council,31
9791,Female,100,Monaghan County Council,7


In [5]:
# Creating a pivot table for population analysis by age and sex
df_analysis = pd.pivot_table(df, "VALUE","Single Year of Age", "Sex")

# Removing the Both sexes column, as it is not needed for this analysis
df_analysis.drop(columns=["Both sexes"], inplace=True)

'''
# Saving the analysis to CSV file
df_analysis.to_csv("population_by_age_and_sex.csv")

'''
# Display the analysis
display(df_analysis.head())

# Source: https://realpython.com/how-to-pandas-pivot-table/#performing-more-advanced-aggregations
# Source: https://data.org/wp-content/uploads/2024/08/5.-Gender-Case-Study-through-Python.pdf
# Source: https://www.analyticsvidhya.com/blog/2020/03/pivot-table-pandas-python/

Sex,Female,Male
Single Year of Age,,
0,909.225806,955.161290
1,888.548387,931.451613
2,934.645161,975.354839
3,951.064516,1000.032258
4,961.903226,1022.129032


In [6]:
# We'll get some descriptive statistics for the population by age and sex in Ireland
df_analysis.describe() # Using df_analysis containing population data by age and sex


Sex,Female,Male
count,101.000000,101.000000
mean,831.871607,812.695305
std,399.652916,411.360302
min,10.838710,3.387097
25%,590.129032,563.645161
50%,961.903226,975.290323
75%,1112.548387,1126.645161
max,1409.548387,1343.354839


### **1.1 Weighted mean age (by sex)**


In [7]:
# Calculate weighted average age for each sex
# We need to calculate the weighted average for each sex column
weight_mean_female = (df_analysis.index * df_analysis['Female']).sum() / df_analysis['Female'].sum()
weight_mean_male = (df_analysis.index * df_analysis['Male']).sum() / df_analysis['Male'].sum()


print("Weighted Average Age by Sex:")
print(f"Female: {weight_mean_female:.2f}")
print(f"Male: {weight_mean_male:.2f}")

# Source: https://saturncloud.io/blog/how-to-calculate-weighted-average-using-pandas-dataframe/
# Source: https://www.quora.com/How-do-I-get-the-weighted-mean-in-pandas
# Source: https://stackoverflow.com/questions/26205922/calculate-weighted-average-using-a-pandas-dataframe


Weighted Average Age by Sex:
Female: 38.94
Male: 37.74


### **1.2 The difference between the sexes by age - Not looking at regions**

In [8]:
# We need to focus on the differences in population by sex in Ireland, I am assuming all counties form Ireland,
# and the difference we are looking, is about the differences between the distribution of Females and Males, by age

# Using existing df_analysis, which has the Ireland-wide data by age and sex
pivot = df_analysis.copy()

# Reset index to make 'Single Year of Age' a column
pivot = pivot.reset_index()

# Check for differences between sexes in Ireland by age
pivot['Female - Male'] = pivot['Female'] - pivot['Male']
pivot['Sex Ratio (F/M)'] = pivot['Female'] / pivot['Male']

# Show the results for all ages
display(pivot[['Single Year of Age', 'Male', 'Female', 'Female - Male', 'Sex Ratio (F/M)']].style.format({
    'Single Year of Age': '{:.0f}', 'Male': '{:.2f}', 'Female': '{:.2f}', 'Female - Male': '{:.2f}',
    'Sex Ratio (F/M)': '{:.2f}' }))

# Source: https://realpython.com/pandas-reset-index/
# Source: https://stackoverflow.com/questions/38951345/how-to-get-rid-of-multilevel-index-after-using-pivot-table-pandas
# Source: Claude AI (Used to help format the display, as I wanted specific columns with 2 decimals)


Sex,Single Year of Age,Male,Female,Female - Male,Sex Ratio (F/M)
0,0,955.16,909.23,-45.94,0.95
1,1,931.45,888.55,-42.90,0.95
2,2,975.35,934.65,-40.71,0.96
3,3,1000.03,951.06,-48.97,0.95
4,4,1022.13,961.90,-60.23,0.94
5,5,1054.32,1011.03,-43.29,0.96
6,6,1099.74,1052.32,-47.42,0.96
7,7,1142.87,1082.97,-59.90,0.95
8,8,1170.84,1110.87,-59.97,0.95
9,9,1192.55,1136.35,-56.19,0.95


## **Part 2 - Grouping ages and differences within selected age group (e.g 35)** 

### **2.1 Making a variable that stores age 35 and groups the people within 5 years of that age together**


In [9]:
# Making a variable that stores age 35 and groups the people within 5 years of that age
age_group = 35
age_range = (age_group - 5, age_group + 5)
age_range_pop = pivot[(pivot['Single Year of Age'] >= age_range[0]) & (pivot['Single Year of Age'] <= age_range[1])]

# Reset the index for clarity
age_range_pop = age_range_pop.reset_index(drop=True)

# Display the population data for the selected age range
display(age_range_pop)


# Source: https://stackoverflow.com/questions/58909624/what-is-the-use-of-reset-index-in-pandas
# Source: https://www.geeksforgeeks.org/python/reset-index-in-pandas-dataframe/
# Source: DeepSeek AI (Used to create age range, as I was getting errors using for_in loops)

Sex,Single Year of Age,Female,Male,Female - Male,Sex Ratio (F/M)
0,30,1059.387097,995.419355,63.967742,1.064262
1,31,1087.419355,1039.903226,47.516129,1.045693
2,32,1109.096774,1045.580645,63.516129,1.060747
3,33,1112.548387,1028.645161,83.903226,1.081567
4,34,1170.451613,1068.419355,102.032258,1.095498
5,35,1223.870968,1119.193548,104.677419,1.093529
6,36,1259.032258,1155.741935,103.290323,1.089371
7,37,1264.290323,1175.064516,89.225806,1.075933
8,38,1319.419355,1210.096774,109.322581,1.090342
9,39,1373.935484,1249.967742,123.967742,1.099177


### **2.2 Calculate the population difference between the sexes in that age group**

In [10]:
# Calculate the total population for each sex in the age range (30-40)
tt_female = age_range_pop['Female'].sum()
tt_male = age_range_pop['Male'].sum()
tt_dif = age_range_pop['Female - Male'].sum()

# Display the totals for the 30-40 age group
print(f"Total Female population (30-40): {tt_female:.0f}")
print(f"Total Male population (30-40): {tt_male:.0f}")
print(f"Total difference (Female - Male): {tt_dif:.0f}")

# Calculate the ratio for Male and Females in the 30-40 age group
age_group_ratio = tt_female / tt_male

print(f'The ratio of Females to Males in the 30-40 age group is {age_group_ratio:.2f}')

# Source: https://www.geeksforgeeks.org/python/python-pandas-dataframe-loc/
# Source: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html

Total Female population (30-40): 13371
Total Male population (30-40): 12388
Total difference (Female - Male): 983
The ratio of Females to Males in the 30-40 age group is 1.08


## **Part 3 - Population differences by sexes in selected age group and by regions**

### **3.1 Biggest population difference between the sexes in the age group**

In [11]:
# Filter the original df for the age group 30-40 to get county-level data
age_filter = df[(df['Single Year of Age'] >= 30) & (df['Single Year of Age'] <= 40)]

# Filter out 'Both sexes' before creating pivot table
age_filter_no_both = age_filter[age_filter['Sex'] != 'Both sexes']

# Create a pivot table showing population by county and sex for the age group
coco_age_pivot = age_filter_no_both.pivot_table(values='VALUE', index='Administrative Counties', columns='Sex', 
												   aggfunc='sum', fill_value=0)

# Calculate the difference between Female and Male populations
coco_age_pivot['Difference (F-M)'] = coco_age_pivot['Female'] - coco_age_pivot['Male']

if (coco_age_pivot['Difference (F-M)'] < 0).any():
	negative_counties = coco_age_pivot[coco_age_pivot['Difference (F-M)'] < 0].index.tolist()
	print(f"Please note: Some administrative counties have more males than females in the age group 30-40: {negative_counties}")

# Calculating the ratio
coco_age_pivot['F/M Ratio'] = coco_age_pivot['Female'] / coco_age_pivot['Male']

# Display the pivot table sorted by population difference
display(coco_age_pivot.sort_values(by='Difference (F-M)', ascending=False).style.format("{:.2f}"))

# Source: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html
# Source:https://stackoverflow.com/questions/20119414/define-aggfunc-for-each-values-column-in-pandas-pivot-table
# Source: https://www.geeksforgeeks.org/python/python-pandas-pivot_table/
# Source: Claude AI ('Fix code - to avoid division error')

Please note: Some administrative counties have more males than females in the age group 30-40: ['Dublin City Council']


Sex,Female,Male,Difference (F-M),F/M Ratio
Administrative Counties,,,,
Fingal County Council,29092.00,26150.00,2942.00,1.11
Cork County Council,26545.00,23706.00,2839.00,1.12
South Dublin County Council,26361.00,23637.00,2724.00,1.12
Kildare County Council,20602.00,18671.00,1931.00,1.10
Meath County Council,17715.00,15981.00,1734.00,1.11
Wicklow County Council,11943.00,10338.00,1605.00,1.16
Galway County Council,13904.00,12421.00,1483.00,1.12
Dún Laoghaire Rathdown County Council,18450.00,17074.00,1376.00,1.08
Wexford County Council,12162.00,10824.00,1338.00,1.12


### **3.2 Region With the Largest Difference**

In [12]:
# Create a pivot table where it shows the biggest population difference between the sexes
# in that age group (30 to 40) and add the administrative counties

# Filter out 'Both Sexes' first, then group by Administrative Counties and Sex
coco_group = age_filter_no_both.groupby(['Administrative Counties', 'Sex'])['VALUE'].sum().unstack(fill_value=0) # Unstack to have Sex as columns 

# Show the top 5 Administrative Counties with the biggest difference
coco_group = coco_group[coco_group.index != 'Ireland'] # Remove Ireland from the list
coco_group['Difference'] = abs(coco_group['Female'] - coco_group['Male'])
coco_group ['F/M Ratio'] = coco_group['Female'] / coco_group['Male']

# Display the top 10 counties with the biggest difference
display(coco_group.sort_values(by='F/M Ratio', ascending=False).head(10).style.format("{:.3f}"))

# Display the region with the largest difference
largest_coco_diff = coco_group['F/M Ratio'].idxmax()
print(f"Region with the largest difference between Females and Males in Age Group 30-40: {largest_coco_diff}")

# Source: https://pandas.pydata.org/pandas-docs/version/1.2.2/user_guide/reshaping.html
# Source: https://stackoverflow.com/questions/72495322/faster-alternative-to-groupby-unstack-then-fillna
# Source: https://stackoverflow.com/questions/69139030/why-and-when-should-use-a-stack-and-unstack-methods
# Source: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html
# Source: https://www.geeksforgeeks.org/python/pandas-groupby-unstack/

Sex,Female,Male,Difference,F/M Ratio
Administrative Counties,,,,
Wicklow County Council,11943.000,10338.000,1605.000,1.155
Leitrim County Council,2500.000,2203.000,297.000,1.135
Wexford County Council,12162.000,10824.000,1338.000,1.124
Cork County Council,26545.000,23706.000,2839.000,1.120
Galway County Council,13904.000,12421.000,1483.000,1.119
Kerry County Council,11125.000,9957.000,1168.000,1.117
South Dublin County Council,26361.000,23637.000,2724.000,1.115
Fingal County Council,29092.000,26150.000,2942.000,1.113
Louth County Council,10928.000,9827.000,1101.000,1.112


Region with the largest difference between Females and Males in Age Group 30-40: Wicklow County Council


End